In [1]:
import os
import numpy as np
import random
import warnings
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from skopt import BayesSearchCV
from skopt.space import Integer, Real

# 固定随机种子
os.environ['PYTHONHASHSEED'] = str(1)
np.random.seed(1)
random.seed(1)
warnings.filterwarnings("ignore")

# 1. 数据加载
data = pd.read_excel('dataset.xlsx')
data.rename(columns={"C0": r"C$_0$"}, inplace=True) 

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# 确保数据为数值型
if not np.issubdtype(X.dtypes, np.number):
    X = X.apply(pd.to_numeric, errors='coerce')
if not np.issubdtype(y.dtype, np.number):
    y = pd.to_numeric(y, errors='coerce')

# 检查 NaN/Inf
if X.isnull().any().any() or y.isnull().any():
    raise ValueError("Dataset contains invalid values (NaN or inf). Please check your data.")

# 2. 数据预处理
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.3, random_state=1
)

# 3. 参数优化 (Bayesian Optimization)
lgb_model = LGBMRegressor(random_state=1, verbose=-1)

param_spaces = {
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'max_depth': Integer(-1, 15),  # -1 表示不限制深度
    'num_leaves': Integer(20, 255),
    'n_estimators': Integer(100, 500),
    'min_child_samples': Integer(5, 30),
    'subsample': Real(0.5, 1.0, prior='uniform'),
    'colsample_bytree': Real(0.5, 1.0, prior='uniform'),
    'reg_alpha': Real(0.0, 5.0, prior='uniform'),
    'reg_lambda': Real(0.0, 5.0, prior='uniform')
}

optimizer = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=param_spaces,
    n_iter=32,
    scoring='neg_mean_squared_error',
    cv=5,
    random_state=1
)

optimizer.fit(X_train, y_train)

# 4. 用最优参数训练
best_params = optimizer.best_params_
print(f'Best parameters: {best_params}')
lgb_optimized = LGBMRegressor(**best_params, random_state=1, verbose=-1)
lgb_optimized.fit(X_train, y_train)

# 5. 模型评估
def evaluate_model(model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metrics = {
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train R^2': r2_score(y_train, y_pred_train),
        'Test R^2': r2_score(y_test, y_pred_test),
        'Train MAE': mean_absolute_error(y_train, y_pred_train),
        'Test MAE': mean_absolute_error(y_test, y_pred_test)
    }
    return metrics

results = evaluate_model(lgb_optimized, X_train, X_test, y_train, y_test)
for metric, value in results.items():
    print(f'{metric}: {value}')

# 6. 导出结果
def export_results(y_train, y_pred_train, y_test, y_pred_test, filename='Results_LGBM.xlsx'):
    with pd.ExcelWriter(filename) as writer:
        train_results_df = pd.DataFrame({'y_true': y_train, 'y_pred': y_pred_train})
        test_results_df = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_test})

        train_results_df.to_excel(writer, sheet_name='Train Results', index=False)
        test_results_df.to_excel(writer, sheet_name='Test Results', index=False)

export_results(
    y_train,
    lgb_optimized.predict(X_train),
    y_test,
    lgb_optimized.predict(X_test)
)


Best parameters: OrderedDict([('colsample_bytree', 0.5), ('learning_rate', 0.15906184650535188), ('max_depth', 15), ('min_child_samples', 5), ('n_estimators', 100), ('num_leaves', 255), ('reg_alpha', 5.0), ('reg_lambda', 5.0), ('subsample', 1.0)])
Train RMSE: 17.234724573396466
Test RMSE: 14.258248219982924
Train R^2: 0.8072098363868649
Test R^2: 0.8812505556879215
Train MAE: 12.30109910439851
Test MAE: 11.047187945543014
